In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

In [ ]:
similarities


In [ ]:
embeddings = model.encode(sentences)
len(embeddings[0])

## Load Libraries

In [1]:
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from tqdm.auto import tqdm

/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the Text Chunks

In [2]:
# 1. Load chunk data
with open('../precessed_data/chunks/text_chunks_v1.json', 'r') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks.")

Loaded 6974 chunks.


## Create Connection and Collection

In [3]:
# 2. Initialize Qdrant Client
client = QdrantClient(host="localhost", port=6333)
collection_name = "research_papers_v1"

# 3. Create collection if it doesn't exist
# We set size=1024 because mixedbread-ai/mxbai-embed-large-v1 outputs 1024-dimensional vectors
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )
    print(f"Created collection '{collection_name}'")

Created collection 'research_papers_v1'


## Define Embedding model

## Form embeddings and add in qdrant

In [8]:
# 4. Embed and upload in batches
batch_size = 64
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
print("Model Loaded.")
for i in tqdm(range(0, len(chunks), batch_size), desc="Uploading to Qdrant"):
    batch = chunks[i:i+batch_size]
    # print(batch[0])
    # break
    texts = [c['page_content'] for c in batch]
    
    # Generate embeddings using the model initialized in the first cell
    embeddings = model.encode(texts)
    
    batch_points = []
    for j, chunk in enumerate(batch):
        meta = chunk.get('metadata', {}).copy()
        
        # Remove large image base64 data from payload to save space
        if 'images' in meta:
            del meta['images']
            
        batch_points.append(
            PointStruct(
                id=i+j,
                vector=embeddings[j].tolist(),
                payload={
                    "text": chunk['page_content'],
                    **meta,
                    "chunk_id": chunk.get('chunk_id', ''),
                    "chunk_index": chunk.get('chunk_index', 0)
                }
            )
        )
    
    client.upsert(
        collection_name=collection_name,
        points=batch_points
    )

print("Finished uploading to Qdrant!")


Model Loaded.


Uploading to Qdrant: 100%|██████████| 109/109 [3:30:07<00:00, 115.66s/it] 

Finished uploading to Qdrant!


Latency = 210m on cpu

# Image Embeddings

In [1]:
import json
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from tqdm.auto import tqdm

/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1782138003.880119   52350 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782138003.925033   52350 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782138005.155987   52350 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to

In [2]:
# 1. Load image chunks data
with open('../precessed_data/chunks/image_chunks_v1.json', 'r') as f:
    image_chunks = json.load(f)

print(f"Loaded {len(image_chunks)} image chunks.")

Loaded 756 image chunks.


In [3]:




# 2. Initialize CLIP model 
# We use clip-ViT-B-32 which outputs 512-dimensional vectors
clip_model = SentenceTransformer('clip-ViT-B-32', device='cpu')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:


# 3. Initialize Qdrant Client
client = QdrantClient(host="localhost", port=6333)
collection_name = "research_papers_v1"

In [5]:


# 4. Embed and upload in batches
batch_size = 32

for i in tqdm(range(0, len(image_chunks), batch_size), desc="Uploading Images to Qdrant"):
    batch = image_chunks[i:i+batch_size]
    
    # Load images using PIL
    images = []
    valid_batch = []
    for chunk in batch:
        try:
            img = Image.open(chunk['image_path'])
            images.append(img)
            valid_batch.append(chunk)
        except Exception as e:
            print(f"Error loading {chunk['image_path']}: {e}")
            continue
            
    if not images:
        continue
        
    # Generate embeddings
    embeddings = clip_model.encode(images)
    
    batch_points = []
    for j, chunk in enumerate(valid_batch):
        # Zero-pad the 512-dim CLIP vector to 1024-dim to fit the existing collection
        padded_vector = np.pad(embeddings[j], (0, 1024 - len(embeddings[j])), 'constant').tolist()
        
        # Create a unique ID to avoid overwriting text chunks. 
        # We start from a high offset (1,000,000) for safety.
        unique_id = 1000000 + i + j
        
        batch_points.append(
            PointStruct(
                id=unique_id,
                vector=padded_vector,
                payload={
                    "image_path": chunk['image_path'],
                    **chunk['metadata'],
                    "chunk_id": chunk.get('chunk_id', ''),
                    "type": "image_chunk" # Add a type to filter easily later
                }
            )
        )
    
    client.upsert(
        collection_name=collection_name,
        points=batch_points
    )

print("Finished uploading image embeddings to Qdrant!")


Uploading Images to Qdrant: 100%|██████████| 24/24 [00:38<00:00,  1.59s/it]

Finished uploading image embeddings to Qdrant!
